In [6]:
import pandas as pd
import numpy as np

df = pd.read_csv('Titanic-Dataset.csv')

In [7]:
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [8]:
# Age: Fill with the median (robust against outliers)
df['Age'] = df['Age'].fillna(df['Age'].median())

# Fare: Only 1 missing value in the test set; fill with median
df['Fare'] = df['Fare'].fillna(df['Fare'].median())

# Embarked: Only 2 missing values; fill with the mode (most frequent port)
embarked_mode = df['Embarked'].mode()[0]
df['Embarked'] = df['Embarked'].fillna(embarked_mode)

# Cabin: Highly missing (~77%). We will fill it with a global constant.
df['Cabin'] = df['Cabin'].fillna('Unknown')

print("Missing values after cleaning:\n", df[['Age', 'Fare', 'Embarked', 'Cabin']].isnull().sum())

Missing values after cleaning:
 Age         0
Fare        0
Embarked    0
Cabin       0
dtype: int64


In [9]:
# Ensure no Age or Fare is negative (fixing potential noise)
df.loc[df['Age'] < 0, 'Age'] = df['Age'].median()
df.loc[df['Fare'] < 0, 'Fare'] = 0

# Extracting just the first letter of the Cabin to reduce noise/complexity
df['Cabin_Deck'] = df['Cabin'].str[0]

In [10]:
# Calculate Q1, Q3, and IQR for Fare
Q1 = df['Fare'].quantile(0.25)
Q3 = df['Fare'].quantile(0.75)
IQR = Q3 - Q1

# Define bounds
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# Cap the outliers (bringing extreme values down to the upper bound)
df['Fare'] = np.clip(df['Fare'], lower_bound, upper_bound)

print(f"Fare capped between {lower_bound} and {upper_bound}")

Fare capped between -26.724 and 65.6344


In [11]:
# Min-Max Normalization for Age (scales between 0 and 1)
age_min = df['Age'].min()
age_max = df['Age'].max()
df['Age_MinMax'] = (df['Age'] - age_min) / (age_max - age_min)

# Z-Score Standardization for Fare (mean=0, std=1)
fare_mean = df['Fare'].mean()
fare_std = df['Fare'].std()
df['Fare_Zscore'] = (df['Fare'] - fare_mean) / fare_std

print(df[['Age', 'Age_MinMax', 'Fare', 'Fare_Zscore']].head())

    Age  Age_MinMax     Fare  Fare_Zscore
0  22.0    0.271174   7.2500    -0.820092
1  38.0    0.472229  65.6344     2.030483
2  26.0    0.321438   7.9250    -0.787135
3  35.0    0.434531  53.1000     1.418500
4  35.0    0.434531   8.0500    -0.781032


In [12]:
# Label Encoding for Sex (Binary)
df['Sex'] = df['Sex'].map({'male': 0, 'female': 1})

# One-Hot Encoding for Embarked and Pclass (Nominal/Categorical)
# This creates dummy variables (e.g., Embarked_S, Embarked_C)
df = pd.get_dummies(df, columns=['Embarked', 'Pclass'], drop_first=True)

# Convert boolean columns to integers (0 and 1) if get_dummies outputted booleans
bool_cols = df.select_dtypes(include='bool').columns
df[bool_cols] = df[bool_cols].astype(int)

print("Encoded Columns Sample:")
print(df.filter(regex='Sex|Embarked|Pclass').head())

Encoded Columns Sample:
   Sex  Embarked_Q  Embarked_S  Pclass_2  Pclass_3
0    0           0           1         0         1
1    1           0           0         0         0
2    1           0           1         0         1
3    1           0           1         0         0
4    0           0           1         0         1


In [13]:
# Binning Age into discrete categories
age_bins = [0, 12, 18, 60, 120]
age_labels = ['Child', 'Teenager', 'Adult', 'Senior']

df['Age_Group'] = pd.cut(df['Age'], bins=age_bins, labels=age_labels, right=False)

print(df[['Age', 'Age_Group']].head(10))

    Age Age_Group
0  22.0     Adult
1  38.0     Adult
2  26.0     Adult
3  35.0     Adult
4  35.0     Adult
5  28.0     Adult
6  54.0     Adult
7   2.0     Child
8  27.0     Adult
9  14.0  Teenager
